# MAT benchmark - Tier 2 on Colab GPU (pretrained MAT vs scratch MAT, 42M parameters)

**Runtime -> Change runtime type -> GPU (T4 is enough).**

This notebook runs only the compute-heavy part of the project (PRD section 3.3, Tier 2). Everything else runs locally. Steps: (1) upload `molbench_colab_bundle.zip` created by `python scripts/make_colab_bundle.py` on the local machine, (2) run all cells, (3) download `results/raw/tier2_runs.csv` and copy it into the local project's `results/raw/` folder.

In [ ]:
!nvidia-smi
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

In [ ]:
!pip install -q rdkit torch_geometric gdown

## 1. Upload the bundle
Either upload `molbench_colab_bundle.zip` directly (cell below) or place it in Google Drive and mount the drive.

In [ ]:
from google.colab import files
up = files.upload()  # choose molbench_colab_bundle.zip
import os
assert 'molbench_colab_bundle.zip' in up, 'upload molbench_colab_bundle.zip'
!rm -rf /content/molbench && mkdir -p /content/molbench && unzip -q -o molbench_colab_bundle.zip -d /content/molbench
%cd /content/molbench
!ls

In [ ]:
%cd /content/molbench
!pip install -q -e .
import molbench, os
print('molbench', molbench.__version__)
print('processed files:', sorted(os.listdir('data/processed'))[:10])

## 2. Pretrained MAT checkpoint (authors' release, Google Drive)

In [ ]:
%cd /content/molbench
!mkdir -p data/pretrained
!gdown 11-TZj8tlnD7ykQGliO9bCrySJNBnYD2k -O data/pretrained/mat_pretrained_weights.pt
import os; print(os.path.getsize('data/pretrained/mat_pretrained_weights.pt')//1_000_000, 'MB')

## 3. Fidelity check (pretrained checkpoint loads completely)

In [ ]:
%cd /content/molbench
!python -m pytest tests/test_mat_fidelity.py -q

## 4. Run Tier 2 (5 seeds x fold 0 x 2 splits x 7 tasks x 2 variants = 140 runs)
The CSV is resume-safe: re-running the cell continues where it stopped.

In [ ]:
%cd /content/molbench
!MOLBENCH_DEVICE=cuda python scripts/04_run_tier2.py --workers 1 --threads 4

## 5. Download the results and copy them into the local project's `results/raw/`

In [ ]:
import pandas as pd
df = pd.read_csv('results/raw/tier2_runs.csv')
print(len(df), 'rows;', int((df['error'].fillna('')!='').sum()), 'errors')
print(df.groupby(['model','split'])[['roc_auc','rmse']].mean())
from google.colab import files
files.download('results/raw/tier2_runs.csv')